In [1]:
import ase
import numpy as np
import pyscf
import time
import os
from pyscf import gto, dft
from pyscf.scf import hf
hf.MUTE_CHKFILE = True
import equiv_dens.utils.base as utils

/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


Use "numpy" for Fourier Transform


In [2]:
# load data

atom_pos = np.load('datasets/resorcinol_atom_pos.npy', allow_pickle=True)

atom_pos = utils.bohr_to_angstrom(atom_pos[-1000:])
print(atom_pos.shape)
atom_types = np.load('datasets/resorcinol_atom_numbers.npy', allow_pickle=True)[0]

(1000, 14, 3)


In [3]:
# test different basis sets
basis_sets = ['631gs', '631gss', 'def2svp', 'ccpvdz', 'aug-ccpvdz']

for basis in basis_sets:
    print('basis', basis)
    start = time.time()
    pos = atom_pos[0]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis=basis)
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    #mf.max_cycle = 1000
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)

basis 631gs
converged SCF energy = -382.20192390971
--------------- RKS gradients ---------------
         x                y                z
0 C    -0.0419491622    -0.0040150026    -0.0136743484
1 C     0.0579201229    -0.0288297632     0.0002914244
2 C    -0.0141669495     0.0025472782    -0.0011761048
3 C     0.0150041178     0.0298555786    -0.0106587372
4 C    -0.0008975161    -0.0017082110     0.0109924646
5 C    -0.0272201069     0.0630999558    -0.0024647454
6 O    -0.0227095836     0.0137416516     0.0047158977
7 H     0.0025835607    -0.0111803580    -0.0002500753
8 O    -0.0311768689    -0.0307622036     0.0019842040
9 H     0.0305279331     0.0259508303    -0.0014207827
10 H     0.0040184227    -0.0099052451     0.0035764197
11 H     0.0150840530     0.0051705017     0.0030838244
12 H    -0.0031180595    -0.0348575183     0.0011036694
13 H     0.0161053209    -0.0191381781     0.0039235491
----------------------------------------------
elapsed 27.609665870666504
basis 631

In [3]:

save_path = 'datasets/resorcinol_pyscf_augccpvdz_dft_test.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), len(atom_pos)):
    print('calc', i)
    start = time.time()
    pos = atom_pos[i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis='augccpvdz')
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    #mf.max_cycle = 1000
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    print('mo occ', mf.mo_occ)
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = forces
    res.append(calc_dict)
    results.append(res)
    
    if i%10 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)

results len 751
calc 751
converged SCF energy = -382.270544022671
--------------- RKS gradients ---------------
         x                y                z
0 C     0.0539304595     0.0036205204     0.0085332158
1 C    -0.0541543672    -0.0424839929     0.0030135394
2 C     0.0363066268     0.0250591304    -0.0205269593
3 C     0.0269322482    -0.0004983711     0.0066646870
4 C     0.0198491640     0.0119665810     0.0075273333
5 C    -0.0294006648     0.0776329203    -0.0109704138
6 O    -0.0676432222    -0.0409562663     0.0041956424
7 H    -0.0060323670    -0.0050242279    -0.0033030100
8 O     0.0200289160     0.0086882801     0.0072434940
9 H    -0.0023017896    -0.0013810482     0.0019200070
10 H     0.0071499812    -0.0173440653    -0.0029782868
11 H     0.0103238052     0.0062414324    -0.0050084458
12 H    -0.0024417894    -0.0228599086     0.0029642208
13 H    -0.0125066070    -0.0027002986     0.0007285740
----------------------------------------------
elapsed 269.3886406421

In [8]:
atom_pos = np.load('datasets/resorcinol_atom_pos.npy', allow_pickle=True)

atom_pos = utils.bohr_to_angstrom(atom_pos[:1004])
print(atom_pos.shape)
atom_types = np.load('datasets/resorcinol_atom_numbers.npy', allow_pickle=True)[0]
load_path = 'datasets/resorcinol_pyscf_augccpvdz_dft_train.npy'
pyscf_data = np.load(load_path, allow_pickle=True)
print('pyscf data len', len(pyscf_data))
data = {}
data['positions'] = atom_pos
data['atom_numbers'] = atom_types
data['atom_types'] = utils.numbers_to_symbols(atom_types)
data['energy'] = []
data['forces'] = []
for calc in pyscf_data:
    data['energy'].append(calc[1]['energy'])
    data['forces'].append(-calc[1]['forces']*utils.to_bohr)
    
data['energy'] = np.array(data['energy'])[:, None]
data['forces'] = np.array(data['forces'])
print('energy shape', data['energy'].shape)
print('forces shape', data['forces'].shape)
save_path = 'datasets/resorcinol_augccpvdz_train.npy'
np.save(save_path, data, allow_pickle=True)

(1004, 14, 3)
pyscf data len 1004
energy shape (1004, 1)
forces shape (1004, 14, 3)


In [14]:
# load data
data = np.load('datasets/resorcinol_combo_kmeansidx-1000_train.npy', allow_pickle=True).item()
atom_pos = data['positions'] 

atom_types = data['atom_numbers'][0]

In [4]:
save_path = 'datasets/resorcinol_just_checking.npy'
if os.path.exists(save_path):
    results = list(np.load(save_path, allow_pickle=True))
else:
    results = []
print('results len', len(results))
for i in range(len(results), 10):
    print('calc', i)
    start = time.time()
    pos = atom_pos[i]
    atom = []
    for j in range(len(atom_types)):
        atom.append((atom_types[j], pos[j, :])) 
    mol = gto.M(atom=atom, basis='augccpvdz')
    #print(mol.pack())
    mf = dft.RKS(mol)
    mf.chkfile=False
    mf.xc = 'pbe'
    #mf.max_cycle = 1000
    mf.kernel()
    g = mf.nuc_grad_method()
    forces = g.grad()
    print('elapsed', time.time() - start)
    #print(mfs[i].mo_coeff)
    res = []
    res.append(mol.pack())
    calc_dict = {}
    print('mo occ', mf.mo_occ)
    calc_dict['mo_coeff'] = mf.mo_coeff
    calc_dict['mo_occ'] = mf.mo_occ
    calc_dict['energy'] = mf.e_tot
    calc_dict['forces'] = forces
    res.append(calc_dict)
    results.append(res)
    
    if i%10 == 0:
        np.save(save_path, results, allow_pickle=True)
np.save(save_path, results, allow_pickle=True)

results len 0
calc 0
converged SCF energy = -382.300554053344
--------------- RKS gradients ---------------
         x                y                z
0 C    -0.0044976071    -0.0041697793     0.0000021162
1 C    -0.0000001159    -0.0015969781     0.0000089104
2 C     0.0044977353    -0.0041704939     0.0000020290
3 C     0.0016491273     0.0021017461    -0.0000016224
4 C     0.0000001671    -0.0002869156    -0.0000031935
5 C    -0.0016491180     0.0021006419    -0.0000016447
6 O    -0.0020339508    -0.0006554316    -0.0000153682
7 H    -0.0007572405     0.0000940983     0.0000116155
8 O     0.0020338196    -0.0006553233    -0.0000153177
9 H     0.0007572615     0.0000940940     0.0000116067
10 H    -0.0000000045    -0.0075332030     0.0000009757
11 H     0.0063807013     0.0035729494    -0.0000009235
12 H     0.0000000539     0.0075204954     0.0000013488
13 H    -0.0063808293     0.0035730676    -0.0000009175
----------------------------------------------
elapsed 129.61975693702698

In [11]:
res_les = np.load('datasets/resorcinol_combo_kmeansidx-1000_train_pyscf_augccpvqzjkfit.npy', allow_pickle=True)

print('leslie energies', [res_les[i][1]['energy'] for i in range(10)])
print('current energies', [results[i][1]['energy'] for i in range(10)])

leslie energies [-382.3005540533428, -382.3005392350197, -382.3005382438111, -382.29945637246345, -382.27893024362766, -382.28222949223823, -382.2874557981481, -382.2794123999044, -382.2756454660915, -382.279751895398]
current energies [-382.30055405334406, -382.3005392350205, -382.3005382438144, -382.2994563724653, -382.27893024355876, -382.2822294921075, -382.28745579796345, -382.27941239992157, -382.27564546610034, -382.2797518953877]


In [12]:
print('leslie forces', [res_les[i][1]['forces'][:, 0] for i in range(10)])
print('current forces', [results[i][1]['forces'][:, 0] for i in range(10)])


leslie forces [array([ 8.49924573e-03,  2.19081901e-07, -8.49948784e-03, -3.11639886e-03,
       -3.15770414e-07,  3.11638128e-03,  3.84360991e-03,  1.43097721e-03,
       -3.84336208e-03, -1.43101689e-03,  8.43173197e-09, -1.20577779e-02,
       -1.01834657e-07,  1.20580198e-02]), array([ 1.15316569e-02, -1.05583410e-03, -8.03075491e-03, -2.37578282e-03,
       -1.11428212e-03,  2.36331203e-03,  2.84889155e-03,  1.05605143e-03,
       -4.34887062e-03, -1.40264953e-03,  5.76202279e-04, -1.20622457e-02,
        5.18796292e-05,  1.21383262e-02]), array([ 7.96854011e-03,  1.16071928e-03, -1.18533974e-02, -2.01677092e-03,
        1.11343489e-03,  2.03376026e-03,  4.00660174e-03,  1.92884822e-03,
       -2.87188485e-03, -9.58187689e-04, -6.19940186e-04, -1.22154327e-02,
       -2.38855016e-06,  1.21499602e-02]), array([ 1.09321943e-02,  1.33835570e-07, -1.09323619e-02, -1.71704886e-03,
        3.69751889e-08,  1.71636722e-03,  3.24599914e-03,  9.53732256e-04,
       -3.24573590e-03, -9.5369

In [13]:

les_en = np.array([res_les[i][1]['energy'] for i in range(10)])
curr_en = np.array([results[i][1]['energy'] for i in range(10)])

les_f = np.array([res_les[i][1]['forces'][:, 0] for i in range(10)])
curr_f = np.array([results[i][1]['forces'][:, 0] for i in range(10)])

print(les_en/curr_en)
print(les_f/curr_f)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
[[-1.88972613 -1.88973671 -1.88972613 -1.88972611 -1.88970931 -1.88972612
  -1.88972612 -1.88972612 -1.88972612 -1.88972612 -1.88970607 -1.88972612
  -1.88972629 -1.88972612]
 [-1.88972612 -1.88972612 -1.88972612 -1.88972612 -1.88972612 -1.88972612
  -1.88972612 -1.88972612 -1.88972612 -1.88972612 -1.88972612 -1.88972612
  -1.88972612 -1.88972612]
 [-1.88972612 -1.88972613 -1.88972612 -1.88972612 -1.88972614 -1.88972612
  -1.88972612 -1.88972612 -1.88972613 -1.88972612 -1.88972613 -1.88972613
  -1.8897263  -1.88972612]
 [-1.88972612 -1.88972683 -1.88972612 -1.88972612 -1.88970116 -1.88972612
  -1.88972612 -1.88972612 -1.88972612 -1.88972612 -1.88972891 -1.88972612
  -1.88972579 -1.88972612]
 [-1.88972612 -1.88972612 -1.88972613 -1.88972612 -1.88972613 -1.88972611
  -1.88972613 -1.88972613 -1.88972613 -1.88972344 -1.88972612 -1.88972612
  -1.88972613 -1.88972613]
 [-1.8897261  -1.88972612 -1.88972612 -1.88972614 -1.88972612 -1.88972613
  -1.88972613 -1.88

In [16]:
print('leslie forces', les_f)
print('data forces', data['forces'][:10, :, 0])

leslie forces [[ 8.49924573e-03  2.19081901e-07 -8.49948784e-03 -3.11639886e-03
  -3.15770414e-07  3.11638128e-03  3.84360991e-03  1.43097721e-03
  -3.84336208e-03 -1.43101689e-03  8.43173197e-09 -1.20577779e-02
  -1.01834657e-07  1.20580198e-02]
 [ 1.15316569e-02 -1.05583410e-03 -8.03075491e-03 -2.37578282e-03
  -1.11428212e-03  2.36331203e-03  2.84889155e-03  1.05605143e-03
  -4.34887062e-03 -1.40264953e-03  5.76202279e-04 -1.20622457e-02
   5.18796292e-05  1.21383262e-02]
 [ 7.96854011e-03  1.16071928e-03 -1.18533974e-02 -2.01677092e-03
   1.11343489e-03  2.03376026e-03  4.00660174e-03  1.92884822e-03
  -2.87188485e-03 -9.58187689e-04 -6.19940186e-04 -1.22154327e-02
  -2.38855016e-06  1.21499602e-02]
 [ 1.09321943e-02  1.33835570e-07 -1.09323619e-02 -1.71704886e-03
   3.69751889e-08  1.71636722e-03  3.24599914e-03  9.53732256e-04
  -3.24573590e-03 -9.53698535e-04  3.51575890e-08 -1.21778720e-02
  -4.01932528e-08  1.21782602e-02]
 [ 3.61548037e-02 -4.31579510e-02 -6.06689896e-03  2.3